# Census Age Data Cleaning — Niagara

## Statistics Canada Census Age Dataset

This notebook documents the cleaning and validation of the Statistics Canada Census Age dataset for Niagara Census Division and its 12 municipalities.

### Raw Dataset
`98100022.csv`

### Cleaned Dataset
`census_age_niagara_cleaned.csv`

### Census Years
- 2016
- 2021

### Main Tasks
- Validate the raw dataset
- Select the correct Census years
- Filter Niagara geography
- Validate DGUID
- Review and preserve the age hierarchy
- Preserve suppression and status information
- Validate missing values
- Validate duplicates
- Validate Niagara geographies
- Save the cleaned dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/raw/98100022.csv")

print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Raw dataset shape: (1399808, 12)

Columns:
['REF_DATE', 'GEO', 'DGUID', 'Census year (2)', 'Age (in single years), average age and median age (128)', 'Coordinate', 'Gender (3a):Total - Gender[1]', 'Symbol', 'Gender (3a):Men+[2]', 'Symbol.1', 'Gender (3a):Women+[3]', 'Symbol.2']


## 1. Initial Validation

The raw dataset was inspected before any filtering or cleaning.

The initial validation checks the dataset dimensions, Census years, and geographic coverage.

In [3]:
print("Dataset shape:", df.shape)

print("\nCensus years:")
print(df['Census year (2)'].value_counts().sort_index())

print("\nNumber of unique geographies:")
print(df['GEO'].nunique())

print("\nNumber of unique DGUIDs:")
print(df['DGUID'].nunique())

Dataset shape: (1399808, 12)

Census years:
Census year (2)
2016    699904
2021    699904
Name: count, dtype: int64

Number of unique geographies:
5107

Number of unique DGUIDs:
5468


## 2. Prepare Niagara Geography

The analysis is restricted to Niagara Census Division and its 12 municipalities.

DGUID is retained as the reliable geographic identifier.

In [4]:
# Step 5 — Prepare the Niagara Geography

niagara_dguids = [
    '2021A00033526',      # Niagara Census Division
    '2021A00053526003',   # Fort Erie
    '2021A00053526011',   # Port Colborne
    '2021A00053526014',   # Wainfleet
    '2021A00053526021',   # West Lincoln
    '2021A00053526028',   # Pelham
    '2021A00053526032',   # Welland
    '2021A00053526037',   # Thorold
    '2021A00053526043',   # Niagara Falls
    '2021A00053526047',   # Niagara-on-the-Lake
    '2021A00053526053',   # St. Catharines
    '2021A00053526057',   # Lincoln
    '2021A00053526065'    # Grimsby
]

niagara_df = df[df['DGUID'].isin(niagara_dguids)].copy()

# Clean whitespace from the original age column
age_col = 'Age (in single years), average age and median age (128)'
niagara_df[age_col] = niagara_df[age_col].str.strip()

# Validate the geography filter
print("Shape:", niagara_df.shape)
print("Unique DGUIDs:", niagara_df['DGUID'].nunique())
print("Unique geographies:", niagara_df['GEO'].nunique())
print(niagara_df[['GEO', 'DGUID']].drop_duplicates().sort_values('GEO').to_string(index=False))

Shape: (3328, 12)
Unique DGUIDs: 13
Unique geographies: 13
                GEO            DGUID
          Fort Erie 2021A00053526003
            Grimsby 2021A00053526065
            Lincoln 2021A00053526057
            Niagara    2021A00033526
      Niagara Falls 2021A00053526043
Niagara-on-the-Lake 2021A00053526047
             Pelham 2021A00053526028
      Port Colborne 2021A00053526011
     St. Catharines 2021A00053526053
            Thorold 2021A00053526037
          Wainfleet 2021A00053526014
            Welland 2021A00053526032
       West Lincoln 2021A00053526021


In [5]:
age_col = 'Age (in single years), average age and median age (128)'

print(
    "Age labels with leading/trailing whitespace:",
    niagara_df[age_col].ne(niagara_df[age_col].str.strip()).sum()
)

Age labels with leading/trailing whitespace: 0


## 3. Census-Year Validation

The Census Age dataset contains two Census periods: 2016 and 2021.

Both Census years are retained. No annual values are created, interpolated, or forecasted.

In [6]:
print("Census years in Niagara dataset:")
print(niagara_df['Census year (2)'].value_counts().sort_index())

Census years in Niagara dataset:
Census year (2)
2016    1664
2021    1664
Name: count, dtype: int64


## 4. Age-Category Review

The age variable contains 128 labels representing different levels of age information.

These include single-year ages, age groups, broad age groups, total age, average age, median age, and other nested categories.

The categories overlap and are therefore not summed together.

In [7]:
age_col = 'Age (in single years), average age and median age (128)'

print("Number of unique age labels:")
print(niagara_df[age_col].nunique())

print("\nAge labels:")
print(niagara_df[age_col].unique())


Number of unique age labels:
128

Age labels:
<StringArray>
[       'Total - Age',      '0 to 14 years',       '0 to 4 years',
       'Under 1 year',                  '1',                  '2',
                  '3',                  '4',       '5 to 9 years',
                  '5',
 ...
                 '94',     '95 to 99 years',                 '95',
                 '96',                 '97',                 '98',
                 '99', '100 years and over',        'Average age',
         'Median age']
Length: 128, dtype: str


In [8]:
def classify_age_category(age):
    age = str(age).strip()

    if age == 'Average age':
        return 'Average age'
    if age == 'Median age':
        return 'Median age'
    if age == 'Total - Age':
        return 'Total age'
    if age == 'Under 1 year':
        return 'Under 1 year'
    if age.isdigit():
        return 'Single-year age'

    broad_groups = [
        '0 to 14 years',
        '15 to 64 years',
        '65 years and over',
        '85 years and over',
        '100 years and over'
    ]

    if age in broad_groups:
        return 'Broad age group'

    if ' to ' in age and age.endswith(' years'):
        return 'Age group'

    return 'Other'

In [9]:
age_category_check = niagara_df[age_col].apply(classify_age_category)

print(age_category_check.value_counts())

Age (in single years), average age and median age (128)
Single-year age    2574
Age group           520
Broad age group     130
Total age            26
Under 1 year         26
Average age          26
Median age           26
Name: count, dtype: int64


## 5. Suppression and Status-Code Validation

Statistics Canada uses status symbols to identify observations requiring special treatment.

The `x` status represents suppressed observations. The `...` status is also preserved as provided by Statistics Canada.

Status-coded observations are not replaced with zero, estimated, or imputed values.

The numeric values and corresponding Symbol fields are retained.

In [10]:
# Check gender status symbols in the Niagara dataset

symbol_cols = [
    'Symbol',
    'Symbol.1',
    'Symbol.2'
]

for col in symbol_cols:
    print(f"\n{col}")
    print(niagara_df[col].value_counts(dropna=False))


Symbol
Symbol
NaN    3328
Name: count, dtype: int64

Symbol.1
Symbol.1
NaN    3328
Name: count, dtype: int64

Symbol.2
Symbol.2
NaN    3328
Name: count, dtype: int64


In [11]:
for col in symbol_cols:
    print(f"{col}:")
    print("x =", (niagara_df[col] == 'x').sum())
    print("... =", (niagara_df[col] == '...').sum())

Symbol:
x = 0
... = 0
Symbol.1:
x = 0
... = 0
Symbol.2:
x = 0
... = 0


## 6. Gender Field Validation

The dataset contains three gender measures:

- Total-Gender
- Men+
- Women+

The original numeric values were preserved.

No attempt was made to force Men+ and Women+ to sum exactly to Total-Gender because Statistics Canada applies random rounding and confidentiality treatment.

In [12]:
gender_cols = [
    'Gender (3a):Total - Gender[1]',
    'Gender (3a):Men+[2]',
    'Gender (3a):Women+[3]'
]

for col in gender_cols:
    print(f"\n{col}")
    print("Data type:", niagara_df[col].dtype)
    print("Missing values:", niagara_df[col].isna().sum())
    print("Numeric values:", pd.to_numeric(
        niagara_df[col], errors='coerce'
    ).notna().sum())


Gender (3a):Total - Gender[1]
Data type: float64
Missing values: 0
Numeric values: 3328

Gender (3a):Men+[2]
Data type: float64
Missing values: 0
Numeric values: 3328

Gender (3a):Women+[3]
Data type: float64
Missing values: 0
Numeric values: 3328


## 7. Missing-Value Validation

Missing values were checked after filtering to Niagara.

Blank Symbol fields are treated as the absence of a status symbol and are not interpreted as missing population values.

In [13]:
check_cols = [
    'GEO',
    'DGUID',
    'Census year (2)',
    age_col,
    'Gender (3a):Total - Gender[1]',
    'Gender (3a):Men+[2]',
    'Gender (3a):Women+[3]',
    'Symbol',
    'Symbol.1',
    'Symbol.2'
]

print("Missing-value check:\n")

for col in check_cols:
    print(
        f"{col}: "
        f"{niagara_df[col].isna().sum()} missing "
        f"out of {len(niagara_df)}"
    )

Missing-value check:

GEO: 0 missing out of 3328
DGUID: 0 missing out of 3328
Census year (2): 0 missing out of 3328
Age (in single years), average age and median age (128): 0 missing out of 3328
Gender (3a):Total - Gender[1]: 0 missing out of 3328
Gender (3a):Men+[2]: 0 missing out of 3328
Gender (3a):Women+[3]: 0 missing out of 3328
Symbol: 3328 missing out of 3328
Symbol.1: 3328 missing out of 3328
Symbol.2: 3328 missing out of 3328


In [14]:
# Step 12 — Duplicate Validation

print("Exact duplicate rows:", niagara_df.duplicated().sum())

key_cols = [
    'DGUID',
    'Census year (2)',
    age_col
]

print(
    "Duplicate DGUID + Census year + Age records:",
    niagara_df.duplicated(subset=key_cols).sum()
)

Exact duplicate rows: 0
Duplicate DGUID + Census year + Age records: 0


In [15]:
# Step 13 — Geographic Validation

expected_geographies = [
    'Niagara',
    'Fort Erie',
    'Port Colborne',
    'Wainfleet',
    'West Lincoln',
    'Pelham',
    'Welland',
    'Thorold',
    'Niagara Falls',
    'Niagara-on-the-Lake',
    'St. Catharines',
    'Lincoln',
    'Grimsby'
]

actual_geographies = sorted(niagara_df['GEO'].unique())

print("Number of geography levels:", len(actual_geographies))
print("\nGeographies:")
print(actual_geographies)

print("\nAll expected geographies present:",
      set(expected_geographies) == set(actual_geographies))

Number of geography levels: 13

Geographies:
['Fort Erie', 'Grimsby', 'Lincoln', 'Niagara', 'Niagara Falls', 'Niagara-on-the-Lake', 'Pelham', 'Port Colborne', 'St. Catharines', 'Thorold', 'Wainfleet', 'Welland', 'West Lincoln']

All expected geographies present: True


In [16]:
# Step 14 — Validate Niagara Census Totals

total_age = niagara_df[
    niagara_df[age_col] == 'Total - Age'
]

print(
    total_age[
        ['Census year (2)', 'Gender (3a):Total - Gender[1]']
    ].sort_values('Census year (2)')
)

        Census year (2)  Gender (3a):Total - Gender[1]
618368             2016                       447890.0
618624             2016                        30710.0
619136             2016                         6370.0
618880             2016                        18305.0
619904             2016                        52295.0
620160             2016                        18800.0
619648             2016                        17110.0
619392             2016                        14500.0
620928             2016                       133110.0
621184             2016                        23790.0
620672             2016                        17515.0
621440             2016                        27315.0
620416             2016                        88070.0
619008             2021                         6885.0
618240             2021                       477940.0
618752             2021                        20035.0
620032             2021                        23815.0
619776    

In [17]:
# Step 15 — Save Cleaned Niagara Dataset

output_path = "census_age_niagara_cleaned.csv"

niagara_df.to_csv(output_path, index=False)

print("Cleaned dataset saved as:", output_path)
print("Final shape:", niagara_df.shape)

Cleaned dataset saved as: census_age_niagara_cleaned.csv
Final shape: (3328, 12)


In [18]:
# Final validation of saved dataset

final_df = pd.read_csv("census_age_niagara_cleaned.csv")

print("Saved dataset shape:", final_df.shape)
print("Exact duplicates:", final_df.duplicated().sum())

print("\nGeography count:")
print(final_df['GEO'].value_counts().sort_index())

print("\nCensus years:")
print(final_df['Census year (2)'].value_counts().sort_index())

Saved dataset shape: (3328, 12)
Exact duplicates: 0

Geography count:
GEO
Fort Erie              256
Grimsby                256
Lincoln                256
Niagara                256
Niagara Falls          256
Niagara-on-the-Lake    256
Pelham                 256
Port Colborne          256
St. Catharines         256
Thorold                256
Wainfleet              256
Welland                256
West Lincoln           256
Name: count, dtype: int64

Census years:
Census year (2)
2016    1664
2021    1664
Name: count, dtype: int64


In [19]:
age_col = 'Age (in single years), average age and median age (128)'

print("Final shape:", niagara_df.shape)

print(
    "Age labels with whitespace:",
    niagara_df[age_col].ne(niagara_df[age_col].str.strip()).sum()
)

print("Unique DGUIDs:", niagara_df['DGUID'].nunique())

print("Exact duplicate rows:", niagara_df.duplicated().sum())

print(
    "Duplicate DGUID-year-age combinations:",
    niagara_df.duplicated(
        subset=['DGUID', 'Census year (2)', age_col]
    ).sum()
)

print("\nPopulation totals:")
print(
    niagara_df[
        niagara_df[age_col] == 'Total - Age'
    ].groupby(['GEO', 'Census year (2)'])[
        'Gender (3a):Total - Gender[1]'
    ].first()
)

Final shape: (3328, 12)
Age labels with whitespace: 0
Unique DGUIDs: 13
Exact duplicate rows: 0
Duplicate DGUID-year-age combinations: 0

Population totals:
GEO                  Census year (2)
Fort Erie            2016                30710.0
                     2021                32900.0
Grimsby              2016                27315.0
                     2021                28885.0
Lincoln              2016                23790.0
                     2021                25715.0
Niagara              2016               447890.0
                     2021               477940.0
Niagara Falls        2016                88070.0
                     2021                94415.0
Niagara-on-the-Lake  2016                17515.0
                     2021                19090.0
Pelham               2016                17110.0
                     2021                18195.0
Port Colborne        2016                18305.0
                     2021                20035.0
St. Catharines       2

In [20]:
niagara_df.to_csv(
    'census_age_niagara_cleaned.csv',
    index=False
)

print("Corrected CSV saved successfully.")

Corrected CSV saved successfully.
